In [41]:
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel
from typing import Optional
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv

In [42]:
load_dotenv()

True

In [43]:
class ChatQA(BaseModel):
    
    Question : str
    Answer : str | None = None

In [44]:
def chat_with_llm(state: ChatQA) -> dict:
    prompt = PromptTemplate(
        template="""You are a helpful assistant, please reply politely in a short way:
User asked Question: {question}""",
        input_variables=["question"],
    )

    updated_prompt = prompt.invoke({"question": state.Question})

    model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    result = model.invoke(updated_prompt)

    return {"Answer": result.content}

In [45]:
# define graph
graph = StateGraph(ChatQA)

# add nodes
graph.add_node('chatbotQA', chat_with_llm)

# add edges
graph.add_edge(START, 'chatbotQA')
graph.add_edge('chatbotQA', END)


In [46]:
# execute the graph
workflow = graph.compile()
output_state = workflow.invoke({"Question": "what is the capital of india?"})

print(output_state)

{'Question': 'what is the capital of india?', 'Answer': 'The capital of India is New Delhi.'}
